# Практический проект: RAG-чатбот

## Цель проекта

Создадим **RAG-чатбот для документации** который:
- Загружает документы (текстовые файлы)
- Разбивает на chunks (фрагменты)
- Создаёт embeddings (векторные представления)
- Сохраняет в векторную БД (ChromaDB)
- Ищет релевантные фрагменты по запросу
- Генерирует ответ на основе найденных документов

### Архитектура:

```
┌─────────────────────────────────────────────┐
│           1. Подготовка данных              │
│                                             │
│  Документы → Chunks → Embeddings → DB      │
└─────────────────────────────────────────────┘
                     ↓
┌─────────────────────────────────────────────┐
│           2. RAG Pipeline                   │
│                                             │
│  Вопрос → Embedding → Поиск → LLM → Ответ  │
└─────────────────────────────────────────────┘
```

### Java аналогия:

```java
// RAG — это как поиск в базе знаний

class DocumentationChatbot {
    VectorDatabase db;  // ChromaDB
    LLMClient llm;      // DeepSeek
    
    String answer(String question) {
        // 1. Поиск релевантных документов
        List<Document> docs = db.similaritySearch(question, limit=3);
        
        // 2. Формируем контекст
        String context = docs.stream()
            .map(Document::getContent)
            .collect(Collectors.joining("\n"));
        
        // 3. Генерируем ответ
        String prompt = String.format(
            "Контекст: %s\nВопрос: %s\nОтвет:",
            context, question
        );
        
        return llm.generate(prompt);
    }
}

// Похоже на:
// - Elasticsearch + AI
// - Lucene + LLM
// - Поиск + генерация ответа
```

## Установка библиотек

In [ ]:
# Установим всё необходимое
!pip install -q langchain langchain-openai langchain-community langchain-chroma langchain-text-splitters chromadb sentence-transformers python-dotenv

In [2]:
import os
from dotenv import load_dotenv

# Загружаем API ключ
load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

print(f"API Key загружен: {api_key[:10]}..." if api_key else "API Key не найден!")

API Key загружен: sk-***...


## Шаг 1: Создание тестовых документов

Создадим несколько документов с информацией о Python, Java и AI.

In [3]:
# Создадим тестовые документы
documents = [
    {
        "title": "Python Основы",
        "content": """Python - это высокоуровневый язык программирования общего назначения.
        Python был создан Гвидо ван Россумом в 1991 году.
        Python известен своим простым и читаемым синтаксисом.
        Python широко используется в веб-разработке, data science, machine learning и автоматизации.
        Популярные фреймворки: Django, Flask, FastAPI.
        Python использует динамическую типизацию и автоматическое управление памятью."""
    },
    {
        "title": "Java Основы",
        "content": """Java - это объектно-ориентированный язык программирования.
        Java был создан компанией Sun Microsystems в 1995 году.
        Java работает на виртуальной машине JVM (Java Virtual Machine).
        Java широко используется в enterprise приложениях, Android разработке и backend системах.
        Популярные фреймворки: Spring Boot, Hibernate, Apache Kafka.
        Java использует статическую типизацию и строгую объектно-ориентированную модель."""
    },
    {
        "title": "LangChain Основы",
        "content": """LangChain - это фреймворк для разработки приложений с LLM.
        LangChain предоставляет удобные абстракции для работы с языковыми моделями.
        Основные компоненты: Chains, Agents, Memory, Tools.
        LangChain поддерживает различные LLM провайдеры: OpenAI, Anthropic, DeepSeek.
        LangChain упрощает создание RAG систем, агентов и чатботов.
        LCEL (LangChain Expression Language) использует pipe оператор для цепочек."""
    },
    {
        "title": "RAG Технология",
        "content": """RAG (Retrieval Augmented Generation) - это техника для улучшения ответов LLM.
        RAG работает в три этапа: поиск релевантных документов, добавление в промпт, генерация ответа.
        RAG использует векторные базы данных для быстрого поиска похожих документов.
        Embeddings преобразуют текст в векторы чисел.
        Популярные векторные БД: ChromaDB, Pinecone, Weaviate, Qdrant.
        RAG решает проблему устаревания знаний LLM и галлюцинаций."""
    },
    {
        "title": "Spring Framework",
        "content": """Spring - это популярный фреймворк для Java разработки.
        Spring Boot упрощает создание production-ready приложений.
        Основные модули: Spring Core (IoC, DI), Spring MVC, Spring Data, Spring Security.
        Spring использует аннотации: @Component, @Service, @Repository, @Controller.
        Spring Boot предоставляет auto-configuration и встроенный сервер.
        Spring поддерживает различные БД, REST API, микросервисную архитектуру."""
    }
]

print(f"Создано {len(documents)} документов")
for doc in documents:
    print(f"  - {doc['title']}")

Создано 5 документов
  - Python Основы
  - Java Основы
  - LangChain Основы
  - RAG Технология
  - Spring Framework


## Шаг 2: Разбивка на chunks и создание embeddings

**Chunking** — разбивка документов на фрагменты.

Зачем нужны chunks?
- LLM имеют ограничение на контекст (например, 4K токенов)
- Маленькие chunks → точнее поиск
- Большие chunks → больше контекста

**Embeddings** — векторные представления текста.

### Java аналогия:

```java
// Chunking — это как pagination
List<String> chunks = document
    .split("\n")
    .stream()
    .filter(s -> s.length() > 0)
    .collect(Collectors.toList());

// Embeddings — это как хеширование, но сохраняет семантику
float[] embedding = embeddingModel.encode("текст");
// Похожие тексты → близкие векторы
```

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# 1. Конвертируем в LangChain Document
langchain_docs = [
    Document(
        page_content=doc["content"],
        metadata={"title": doc["title"]}
    )
    for doc in documents
]

# 2. Создаём text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,  # Размер chunk в символах
    chunk_overlap=50,  # Перекрытие между chunks (для контекста)
    separators=["\n\n", "\n", ".", " "]  # Разделители (по приоритету)
)

# 3. Разбиваем документы на chunks
chunks = text_splitter.split_documents(langchain_docs)

print(f"\nВсего chunks: {len(chunks)}")
print(f"\nПример chunk:")
print(f"Текст: {chunks[0].page_content}")
print(f"Metadata: {chunks[0].metadata}")

In [ ]:
# 4. Создаём embedding model
# Используем small модель для быстроты
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",  # Легковесная модель
    model_kwargs={'device': 'cpu'}  # CPU (для GPU: 'cuda')
)

print("Embedding model загружен!")

# Проверим embeddings
test_embedding = embeddings.embed_query("Python это язык программирования")
print(f"Размерность вектора: {len(test_embedding)}")
print(f"Первые 5 значений: {test_embedding[:5]}")

## Шаг 3: Создание векторной БД

**ChromaDB** — векторная база данных для хранения embeddings.

### Как работает поиск:

```
1. Вопрос: "Что такое Python?"
   ↓
2. Embedding вопроса: [0.1, 0.5, -0.3, ...]
   ↓
3. Similarity search в векторной БД
   ↓
4. Находит closest vectors (cosine similarity)
   ↓
5. Возвращает топ-3 документа
```

### Java аналогия:

```java
// ChromaDB — это как БД с индексом

// Обычная БД:
@Repository
interface DocRepository extends JpaRepository<Document, Long> {
    List<Document> findByTitleContaining(String keyword);
}

// Векторная БД:
interface VectorRepository {
    List<Document> similaritySearch(float[] queryVector, int limit);
    // Находит документы с близкими векторами
}

// Похоже на:
// - Elasticsearch with vector search
// - PostgreSQL with pgvector extension
```

In [ ]:
from langchain_chroma import Chroma

# Создаём векторную БД из chunks
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="docs_collection"  # Имя коллекции
)

print(f"Векторная БД создана!")
print(f"Всего документов в БД: {vectorstore._collection.count()}")

In [ ]:
# Тестируем поиск
query = "Расскажи про Python"
results = vectorstore.similarity_search(query, k=3)  # Топ-3 результата

print(f"Запрос: {query}")
print(f"\nНайдено {len(results)} релевантных документов:\n")

for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.metadata['title']}")
    print(f"   {doc.page_content[:100]}...\n")

## Шаг 4: Создание RAG Chain

Соберём всё вместе в **RAG Chain**:

```
Вопрос → Retriever → Prompt → LLM → Ответ
```

**Retriever** — компонент который ищет документы.

### Java аналогия:

```java
// RAG Chain — это как Service с несколькими шагами

@Service
class RAGService {
    @Autowired VectorRepository vectorRepo;
    @Autowired LLMClient llmClient;
    
    String answer(String question) {
        // 1. Retrieval
        float[] queryVector = embeddings.encode(question);
        List<Document> docs = vectorRepo.similaritySearch(queryVector, 3);
        
        // 2. Augmentation (добавляем в промпт)
        String context = buildContext(docs);
        String prompt = buildPrompt(context, question);
        
        // 3. Generation
        return llmClient.generate(prompt);
    }
}
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Создаём Retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}  # Возвращать топ-3 документа
)

# 2. Создаём LLM
llm = ChatOpenAI(
    base_url="https://api.deepseek.com",
    api_key=api_key,
    model="deepseek-chat",
    temperature=0
)

# 3. Создаём промпт
template = """Ты помощник который отвечает на вопросы используя предоставленный контекст.
Используй ТОЛЬКО информацию из контекста для ответа.
Если в контексте нет ответа, скажи "Я не нашёл информацию в документах".
Отвечай на русском языке.

Контекст:
{context}

Вопрос: {question}

Ответ:"""

prompt = ChatPromptTemplate.from_template(template)

print("RAG компоненты созданы!")

In [ ]:
# 4. Функция для форматирования документов
def format_docs(docs):
    """Объединяет документы в один текст"""
    return "\n\n".join([f"Документ: {doc.metadata['title']}\n{doc.page_content}" for doc in docs])

# 5. Создаём RAG Chain с LCEL
rag_chain = (
    {
        "context": retriever | format_docs,  # Retriever → format
        "question": RunnablePassthrough()  # Передаём вопрос как есть
    }
    | prompt  # Формируем промпт
    | llm  # Генерируем ответ
    | StrOutputParser()  # Парсим в строку
)

print("RAG Chain создан!")

## Шаг 5: Тестирование RAG чатбота

In [ ]:
# Тест 1: Вопрос про Python
question = "Что такое Python и когда он был создан?"
print(f"Вопрос: {question}\n")

answer = rag_chain.invoke(question)
print(f"Ответ: {answer}")

In [ ]:
# Тест 2: Вопрос про Java
question = "Какие фреймворки используются в Java?"
print(f"Вопрос: {question}\n")

answer = rag_chain.invoke(question)
print(f"Ответ: {answer}")

In [ ]:
# Тест 3: Вопрос про RAG
question = "Как работает RAG?"
print(f"Вопрос: {question}\n")

answer = rag_chain.invoke(question)
print(f"Ответ: {answer}")

In [ ]:
# Тест 4: Вопрос которого нет в базе
question = "Что такое Rust?"
print(f"Вопрос: {question}\n")

answer = rag_chain.invoke(question)
print(f"Ответ: {answer}")

In [ ]:
# Тест 5: Сравнение
question = "В чём разница между Python и Java?"
print(f"Вопрос: {question}\n")

answer = rag_chain.invoke(question)
print(f"Ответ: {answer}")

## Шаг 6: Интерактивный чат

Добавим **историю диалога** для multi-turn разговора.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# Создаём чат с историей
class RAGChatbot:
    def __init__(self, rag_chain, retriever):
        self.rag_chain = rag_chain
        self.retriever = retriever
        self.history = []  # История диалога
    
    def chat(self, question: str) -> str:
        """Отвечает на вопрос используя RAG и историю"""
        print(f"\n{'='*60}")
        print(f"Вы: {question}")
        print(f"{'='*60}")
        
        # Получаем релевантные документы
        docs = self.retriever.invoke(question)
        print(f"\n🔍 Найдено {len(docs)} релевантных документов:")
        for i, doc in enumerate(docs, 1):
            print(f"   {i}. {doc.metadata['title']}")
        
        # Генерируем ответ
        answer = self.rag_chain.invoke(question)
        
        # Сохраняем в историю
        self.history.append({"question": question, "answer": answer})
        
        print(f"\n🤖 Бот: {answer}")
        return answer
    
    def show_history(self):
        """Показывает историю диалога"""
        print("\n📜 История диалога:")
        for i, item in enumerate(self.history, 1):
            print(f"\n{i}. Вы: {item['question']}")
            print(f"   Бот: {item['answer'][:100]}...")

# Создаём чатбот
chatbot = RAGChatbot(rag_chain, retriever)
print("Чатбот готов к работе!")

In [ ]:
# Диалог 1
chatbot.chat("Расскажи про Python")

In [ ]:
# Диалог 2
chatbot.chat("А какие фреймворки для него есть?")

In [ ]:
# Диалог 3
chatbot.chat("Сравни Python и Java по типизации")

In [ ]:
# Показываем историю
chatbot.show_history()

## Улучшения для production

### 1. Лучшие embeddings

```python
# Вместо all-MiniLM-L6-v2 используй:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"  # Или multilingual-e5-large
)
```

### 2. Персистентность векторной БД

```python
# Сохранение в файл
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"  # Папка для сохранения
)

# Загрузка из файла
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)
```

### 3. Более умный chunking

```python
# Semantic chunking - разбивка по смыслу
from langchain.text_splitter import SemanticChunker

text_splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile"  # Разбивает где меняется тема
)
```

### 4. Гибридный поиск

```python
# Комбинация vector search + keyword search
from langchain.retrievers import EnsembleRetriever
from langchain.retrievers import BM25Retriever

# Vector search
vector_retriever = vectorstore.as_retriever()

# Keyword search (BM25)
bm25_retriever = BM25Retriever.from_documents(chunks)

# Ensemble (комбинация)
ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]  # Равный вес
)
```

### 5. Re-ranking

```python
# Переранжирование результатов для лучшей точности
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)
```

### 6. Мониторинг и метрики

```python
# Логирование поисковых запросов
import logging

logger = logging.getLogger(__name__)

def search_with_metrics(query):
    start = time.time()
    results = retriever.get_relevant_documents(query)
    duration = time.time() - start
    
    logger.info(f"Query: {query}, Results: {len(results)}, Time: {duration:.2f}s")
    return results
```

### Java аналогия для production:

```java
@Service
class ProductionRAGService {
    @Autowired VectorRepository vectorRepo;  // PostgreSQL + pgvector
    @Autowired CacheManager cacheManager;    // Redis для кеша
    @Autowired MetricsService metrics;       // Prometheus метрики
    
    @Cacheable("rag-answers")
    @Timed("rag.answer.time")
    String answer(String question) {
        // 1. Search with metrics
        Timer.Sample timer = Timer.start();
        List<Document> docs = vectorRepo.similaritySearch(question);
        timer.stop(metrics.timer("rag.search"));
        
        // 2. Generate with timeout
        CompletableFuture<String> future = CompletableFuture
            .supplyAsync(() -> llm.generate(buildPrompt(docs, question)));
        
        try {
            return future.get(30, TimeUnit.SECONDS);
        } catch (TimeoutException e) {
            return "Timeout";
        }
    }
}
```

## Итоги проекта

### Что мы создали:
- ✅ RAG-чатбот для работы с документацией
- ✅ Векторный поиск (ChromaDB + embeddings)
- ✅ LangChain для структуры
- ✅ Интерактивный чат с историей

### Технологии:
1. **LangChain** — фреймворк для LLM приложений
2. **ChromaDB** — векторная БД
3. **HuggingFace** — embeddings модели
4. **DeepSeek** — LLM для генерации

### Архитектура:
```
Документы → Chunks → Embeddings → ChromaDB
                                      ↓
Вопрос → Embedding → Similarity Search → Топ-3
                                      ↓
                    Prompt + Context → LLM → Ответ
```

### Java аналогия:
```java
// RAG System Architecture

@Service
class RAGChatbotService {
    @Autowired DocumentService documentService;      // Загрузка документов
    @Autowired EmbeddingService embeddingService;    // Генерация embeddings
    @Autowired VectorRepository vectorRepository;    // ChromaDB
    @Autowired LLMService llmService;               // DeepSeek API
    
    // Подготовка данных
    void indexDocuments(List<Document> documents) {
        List<Chunk> chunks = documentService.split(documents);
        List<Embedding> embeddings = embeddingService.embed(chunks);
        vectorRepository.saveAll(embeddings);
    }
    
    // RAG запрос
    String answer(String question) {
        // Retrieval
        Embedding queryEmbedding = embeddingService.embed(question);
        List<Document> relevantDocs = vectorRepository
            .similaritySearch(queryEmbedding, 3);
        
        // Augmentation + Generation
        String context = buildContext(relevantDocs);
        String prompt = buildPrompt(context, question);
        return llmService.generate(prompt);
    }
}
```

### Следующие шаги:
1. **Deployment** — деплой через FastAPI или Streamlit
2. **Мониторинг** — добавить метрики и логирование
3. **Улучшения** — hybrid search, re-ranking
4. **Интеграция** — Telegram бот, Slack, веб-интерфейс